In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

print("Libraries loaded ✓")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

In [ ]:
import os

# Check what's inside raw folder
raw_path = r"D:\COEP\Cricket_Project\data\raw"
contents = os.listdir(raw_path)
print(f"Files in raw folder: {len(contents)}")
print("\nFirst 10 items:")
for item in contents[:10]:
    print(item)

In [ ]:
raw_path = r"D:\COEP\Cricket_Project\data\raw"

# Only ball-by-ball files, not info files
all_files = glob.glob(os.path.join(raw_path, "[!*_info]*.csv"))

# Better way — exclude info files
all_files = [f for f in glob.glob(os.path.join(raw_path, "*.csv")) 
             if "_info" not in f]

print(f"Ball-by-ball match files: {len(all_files)}")
print(f"\nSample file path: {all_files[0]}")

In [ ]:
# Load all matches into one DataFrame
# tqdm gives a progress bar so you can see it working

from tqdm.notebook import tqdm

dfs = []
for f in tqdm(all_files, desc="Loading matches"):
    try:
        df_temp = pd.read_csv(f)
        # Add match_id from filename
        match_id = os.path.basename(f).replace(".csv", "")
        df_temp['match_id'] = match_id
        dfs.append(df_temp)
    except Exception as e:
        print(f"Error in {f}: {e}")

df = pd.concat(dfs, ignore_index=True)

print(f"\n✓ Total deliveries loaded: {len(df):,}")
print(f"✓ Total matches: {df['match_id'].nunique()}")
print(f"✓ Columns: {df.columns.tolist()}")

In [ ]:
# Look at one match completely — understand the structure
sample_match = df[df['match_id'] == df['match_id'].iloc[0]]
print(f"One match has {len(sample_match)} deliveries")
print(f"\nFirst 10 balls of this match:")
sample_match.head(10)

In [ ]:
# Fix season column first
df['season'] = df['season'].astype(str).str[:4].astype(int)

# Now run overview
print("=== DATASET OVERVIEW ===\n")
print(f"Seasons covered: {sorted(df['season'].unique())}")
print(f"\nTotal matches: {df['match_id'].nunique()}")
print(f"Total deliveries: {len(df):,}")
print(f"Unique batters: {df['striker'].nunique()}")
print(f"Unique bowlers: {df['bowler'].nunique()}")
print(f"Unique venues: {df['venue'].nunique()}")
print(f"Unique teams: {df['batting_team'].nunique()}")

print(f"\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# === DATA CLEANING ===

# 1. Drop columns that are 100% empty — useless
df = df.drop(columns=['penalty', 'other_wicket_type', 'other_player_dismissed'])

# 2. Fill NaN in extras columns with 0 — NaN means the event didn't happen
extras_cols = ['wides', 'noballs', 'byes', 'legbyes']
df[extras_cols] = df[extras_cols].fillna(0)

# 3. Extract over number from ball column
# ball column is like 0.1, 0.2 ... 0.6, 1.1 etc.
# 0.1 means over 0, ball 1
df['over_number'] = df['ball'].astype(str).str.split('.').str[0].astype(int)
df['ball_number'] = df['ball'].astype(str).str.split('.').str[1].astype(int)

# 4. Total runs per delivery
df['total_runs'] = df['runs_off_bat'] + df['extras']

# 5. Is this a wicket delivery?
df['is_wicket'] = df['wicket_type'].notna().astype(int)

# 6. Is this a legal delivery? (wides and noballs don't count as legal balls)
df['is_legal'] = ((df['wides'] == 0) & (df['noballs'] == 0)).astype(int)

# 7. Convert date
df['start_date'] = pd.to_datetime(df['start_date'])

print("=== AFTER CLEANING ===")
print(f"Columns remaining: {df.columns.tolist()}")
print(f"\nSample of new columns:")
df[['ball', 'over_number', 'ball_number', 'total_runs', 'is_wicket', 'is_legal']].head(10)

In [ ]:
import os

# Create the directory if it doesn't exist
os.makedirs(r"D:\COEP\Cricket_Project\data\processed", exist_ok=True)

# Now save
save_path = r"D:\COEP\Cricket_Project\data\processed\ipl_cleaned.csv"
df.to_csv(save_path, index=False)

print(f"✓ Saved successfully")
print(f"✓ File size: {os.path.getsize(save_path) / 1024 / 1024:.1f} MB")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load cleaned data — start every future session from here
df = pd.read_csv(r"D:\COEP\Cricket_Project\data\processed\ipl_cleaned.csv",
                 parse_dates=['start_date'])

print(f"✓ Data loaded: {len(df):,} deliveries | {df['match_id'].nunique()} matches")
print(f"✓ Seasons: {sorted(df['season'].unique())}")

In [ ]:
# === INSIGHT 1: Has IPL become more aggressive over the years? ===

# Average runs per over per season
runs_per_over = df[df['is_legal'] == 1].groupby('season').agg(
    total_runs = ('runs_off_bat', 'sum'),
    total_overs = ('over_number', 'nunique')
).reset_index()

# Better approach — economy per season
season_stats = df.groupby('season').agg(
    total_runs = ('total_runs', 'sum'),
    total_matches = ('match_id', 'nunique'),
    total_wickets = ('is_wicket', 'sum')
).reset_index()

season_stats['runs_per_match'] = (season_stats['total_runs'] / season_stats['total_matches']).round(1)
season_stats['wickets_per_match'] = (season_stats['total_wickets'] / season_stats['total_matches']).round(1)

print(season_stats[['season','runs_per_match','wickets_per_match']].to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('IPL Evolution: Has the Game Become More Aggressive?', 
             fontsize=16, fontweight='bold', y=1.01)

# Chart 1: Runs per match trend
ax1.plot(season_stats['season'], season_stats['runs_per_match'], 
         color='#E8612C', linewidth=2.5, marker='o', markersize=6)
ax1.fill_between(season_stats['season'], season_stats['runs_per_match'], 
                 alpha=0.15, color='#E8612C')

# Annotate min and max
max_idx = season_stats['runs_per_match'].idxmax()
min_idx = season_stats['runs_per_match'].idxmin()
ax1.annotate(f"Peak: {season_stats.loc[max_idx,'runs_per_match']}",
             xy=(season_stats.loc[max_idx,'season'], season_stats.loc[max_idx,'runs_per_match']),
             xytext=(10, -20), textcoords='offset points',
             fontsize=9, color='#E8612C', fontweight='bold')
ax1.annotate(f"Low: {season_stats.loc[min_idx,'runs_per_match']}",
             xy=(season_stats.loc[min_idx,'season'], season_stats.loc[min_idx,'runs_per_match']),
             xytext=(10, 10), textcoords='offset points',
             fontsize=9, color='gray')

ax1.set_ylabel('Average Runs Per Match', fontsize=11)
ax1.set_xlabel('')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_xticks(season_stats['season'])
ax1.tick_params(axis='x', rotation=45)

# Chart 2: Wickets per match
ax2.bar(season_stats['season'], season_stats['wickets_per_match'],
        color=['#2ecc71' if w >= 12 else '#95a5a6' for w in season_stats['wickets_per_match']],
        width=0.6, edgecolor='white')
ax2.axhline(y=season_stats['wickets_per_match'].mean(), 
            color='red', linestyle='--', linewidth=1.5, 
            label=f"Average: {season_stats['wickets_per_match'].mean():.1f}")
ax2.set_ylabel('Average Wickets Per Match', fontsize=11)
ax2.set_xlabel('Season', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')
ax2.set_xticks(season_stats['season'])
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(r'D:\COEP\Cricket_Project\outputs\01_ipl_evolution.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Chart saved")

In [ ]:
# === INSIGHT 2: Phase-wise scoring analysis ===

# Define match phases
def get_phase(over):
    if over <= 5:
        return 'Powerplay (0-5)'
    elif over <= 9:
        return 'Middle 1 (6-9)'
    elif over <= 14:
        return 'Middle 2 (10-14)'
    else:
        return 'Death (15-19)'

df['phase'] = df['over_number'].apply(get_phase)

# Phase order for plotting
phase_order = ['Powerplay (0-5)', 'Middle 1 (6-9)', 'Middle 2 (10-14)', 'Death (15-19)']

# Runs and wickets per phase
phase_stats = df.groupby('phase').agg(
    total_runs = ('total_runs', 'sum'),
    total_balls = ('is_legal', 'sum'),
    total_wickets = ('is_wicket', 'sum'),
    total_matches = ('match_id', 'nunique')
).reset_index()

# Economy = runs per 6 legal balls
phase_stats['economy'] = (phase_stats['total_runs'] / phase_stats['total_balls'] * 6).round(2)
phase_stats['wickets_per_match'] = (phase_stats['total_wickets'] / phase_stats['total_matches']).round(2)

# Reorder
phase_stats['phase'] = pd.Categorical(phase_stats['phase'], categories=phase_order, ordered=True)
phase_stats = phase_stats.sort_values('phase')

print(phase_stats[['phase','economy','wickets_per_match']].to_string(index=False))

In [ ]:
# === INSIGHT 3: Death Over Batting Specialists ===

# Only death overs, only legal balls
death = df[(df['over_number'] >= 15) & (df['is_legal'] == 1)].copy()

death_batting = death.groupby('striker').agg(
    runs = ('runs_off_bat', 'sum'),
    balls = ('is_legal', 'sum'),
    sixes = ('runs_off_bat', lambda x: (x == 6).sum()),
    fours = ('runs_off_bat', lambda x: (x == 4).sum()),
    dismissals = ('is_wicket', 'sum')
).reset_index()

# Minimum 150 balls — filters out small sample flukes
death_batting = death_batting[death_batting['balls'] >= 150].copy()

death_batting['death_sr'] = (death_batting['runs'] / death_batting['balls'] * 100).round(1)
death_batting['boundary_pct'] = ((death_batting['sixes'] + death_batting['fours']) / death_batting['balls'] * 100).round(1)
death_batting['six_rate'] = (death_batting['sixes'] / death_batting['balls'] * 100).round(1)

# Top 15 by death SR
top15 = death_batting.nlargest(15, 'death_sr')

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['#E8612C' if sr >= 160 else '#3498db' if sr >= 140 else '#95a5a6' 
          for sr in top15['death_sr']]

bars = ax.barh(top15['striker'], top15['death_sr'], color=colors, edgecolor='white', height=0.6)

# Add value labels on bars
for bar, sr, six_rate in zip(bars, top15['death_sr'], top15['six_rate']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'SR: {sr} | 6s: {six_rate}%',
            va='center', fontsize=9, color='#2c3e50')

ax.axvline(x=death_batting['death_sr'].mean(), color='red', 
           linestyle='--', linewidth=1.5,
           label=f"Average Death SR: {death_batting['death_sr'].mean():.1f}")

ax.set_xlabel('Strike Rate in Death Overs (15-19)', fontsize=11)
ax.set_title('IPL Death Over Batting Specialists\n(Min. 150 legal balls faced in overs 15-19)', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--', axis='x')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(r'D:\COEP\Cricket_Project\outputs\03_death_batting.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nTop 5 Death Over Specialists:")
print(top15[['striker','death_sr','six_rate','boundary_pct','balls']].head())

In [ ]:
# === INSIGHT 4: Powerplay Bowling Specialists ===

# Powerplay only, legal balls only
pp = df[(df['over_number'] <= 5) & (df['is_legal'] == 1)].copy()

pp_bowling = pp.groupby('bowler').agg(
    runs_conceded = ('total_runs', 'sum'),
    balls = ('is_legal', 'sum'),
    wickets = ('is_wicket', 'sum'),
    matches = ('match_id', 'nunique'),
    dot_balls = ('total_runs', lambda x: (x == 0).sum())
).reset_index()

# Minimum 200 powerplay balls bowled
pp_bowling = pp_bowling[pp_bowling['balls'] >= 200].copy()

pp_bowling['economy'] = (pp_bowling['runs_conceded'] / pp_bowling['balls'] * 6).round(2)
pp_bowling['strike_rate'] = (pp_bowling['balls'] / pp_bowling['wickets'].replace(0, 1)).round(1)
pp_bowling['dot_pct'] = (pp_bowling['dot_balls'] / pp_bowling['balls'] * 100).round(1)

# Bowling Impact Score — lower economy + more wickets = better
# Normalize both metrics to 0-1 scale then combine
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# For economy — lower is better, so invert
pp_bowling['economy_score'] = 1 - scaler.fit_transform(pp_bowling[['economy']])
# For wickets per match — higher is better
pp_bowling['wicket_score'] = scaler.fit_transform(
    (pp_bowling['wickets']/pp_bowling['matches']).values.reshape(-1,1))
# Dot ball pressure
pp_bowling['dot_score'] = scaler.fit_transform(pp_bowling[['dot_pct']])

# Combined impact
pp_bowling['bowling_impact'] = (
    0.4 * pp_bowling['economy_score'] +
    0.4 * pp_bowling['wicket_score'] +
    0.2 * pp_bowling['dot_score']
).round(3)

top15_bowl = pp_bowling.nlargest(15, 'bowling_impact')

# Plot — two panels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('IPL Powerplay Bowling Specialists (Min. 200 balls)', 
             fontsize=14, fontweight='bold')

# Left — Bowling Impact Score
colors_impact = ['#2ecc71' if i < 5 else '#3498db' if i < 10 else '#95a5a6' 
                 for i in range(len(top15_bowl))]
ax1.barh(top15_bowl['bowler'], top15_bowl['bowling_impact'], 
         color=colors_impact, edgecolor='white', height=0.6)
ax1.set_xlabel('Bowling Impact Score (0-1)', fontsize=11)
ax1.set_title('Overall Powerplay Impact\n(Economy + Wickets + Dot %)', fontsize=11)
ax1.invert_yaxis()
ax1.grid(True, alpha=0.3, linestyle='--', axis='x')

# Right — Economy vs Wickets scatter
ax2.scatter(pp_bowling['economy'], 
            pp_bowling['wickets']/pp_bowling['matches'],
            s=pp_bowling['dot_pct']*3,
            alpha=0.6, color='#3498db', edgecolors='white', linewidth=0.5)

# Label top 10
top10 = pp_bowling.nlargest(10, 'bowling_impact')
for _, row in top10.iterrows():
    ax2.annotate(row['bowler'].split()[-1],  # last name only
                xy=(row['economy'], row['wickets']/row['matches']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=8, color='#2c3e50')

ax2.set_xlabel('Economy Rate (lower = better)', fontsize=11)
ax2.set_ylabel('Wickets Per Match', fontsize=11)
ax2.set_title('Economy vs Wickets\n(bubble size = dot ball %)', fontsize=11)
ax2.invert_xaxis()  # lower economy on right = visually better bowlers top-right
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(r'D:\COEP\Cricket_Project\outputs\04_powerplay_bowling.png', 
            dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 Powerplay Bowlers:")
print(top10[['bowler','economy','bowling_impact','dot_pct',
             'wickets','matches']].to_string(index=False))

In [ ]:
# === VENUE AUDIT — Find all duplicates ===

all_venues = df['venue'].value_counts().reset_index()
all_venues.columns = ['venue', 'deliveries']
print(f"Total unique venues: {len(all_venues)}")
print("\nAll venues with delivery count:")
print(all_venues.to_string())

In [ ]:
# === COMPLETE VENUE STANDARDIZATION ===

venue_mapping = {
    # Wankhede Stadium, Mumbai
    'Wankhede Stadium, Mumbai': 'Wankhede Stadium',

    # Feroz Shah Kotla → renamed to Arun Jaitley Stadium
    'Feroz Shah Kotla': 'Arun Jaitley Stadium',
    'Arun Jaitley Stadium, Delhi': 'Arun Jaitley Stadium',

    # MA Chidambaram Stadium, Chennai
    'MA Chidambaram Stadium, Chepauk': 'MA Chidambaram Stadium',
    'MA Chidambaram Stadium, Chepauk, Chennai': 'MA Chidambaram Stadium',

    # Rajiv Gandhi International Stadium, Hyderabad
    'Rajiv Gandhi International Stadium, Uppal': 'Rajiv Gandhi Intl Stadium',
    'Rajiv Gandhi International Stadium, Uppal, Hyderabad': 'Rajiv Gandhi Intl Stadium',
    'Rajiv Gandhi International Stadium': 'Rajiv Gandhi Intl Stadium',

    # Eden Gardens, Kolkata
    'Eden Gardens, Kolkata': 'Eden Gardens',

    # M Chinnaswamy Stadium, Bengaluru
    'M Chinnaswamy Stadium, Bengaluru': 'M Chinnaswamy Stadium',
    'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium',

    # Sawai Mansingh Stadium, Jaipur
    'Sawai Mansingh Stadium, Jaipur': 'Sawai Mansingh Stadium',

    # Dr DY Patil Sports Academy, Mumbai
    'Dr DY Patil Sports Academy, Mumbai': 'Dr DY Patil Sports Academy',

    # Brabourne Stadium, Mumbai
    'Brabourne Stadium, Mumbai': 'Brabourne Stadium',

    # Punjab Cricket Association Stadium, Mohali
    'Punjab Cricket Association Stadium, Mohali': 'PCA Stadium Mohali',
    'Punjab Cricket Association IS Bindra Stadium, Mohali': 'PCA Stadium Mohali',
    'Punjab Cricket Association IS Bindra Stadium': 'PCA Stadium Mohali',
    'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh': 'PCA Stadium Mohali',

    # Maharashtra Cricket Association Stadium, Pune
    'Maharashtra Cricket Association Stadium, Pune': 'MCA Stadium Pune',
    'Maharashtra Cricket Association Stadium': 'MCA Stadium Pune',

    # Sardar Patel Stadium → renamed to Narendra Modi Stadium
    'Sardar Patel Stadium, Motera': 'Narendra Modi Stadium',

    # Dr YSR ACA-VDCA Stadium, Visakhapatnam
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 'ACA-VDCA Stadium Vizag',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam': 'ACA-VDCA Stadium Vizag',

    # Himachal Pradesh Cricket Association Stadium, Dharamsala
    'Himachal Pradesh Cricket Association Stadium, Dharamsala': 'HPCA Stadium Dharamsala',
    'Himachal Pradesh Cricket Association Stadium': 'HPCA Stadium Dharamsala',

    # Maharaja Yadavindra Singh Stadium, Mullanpur
    'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur': 'MYS Stadium Mullanpur',
    'Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh': 'MYS Stadium Mullanpur',

    # Subrata Roy Sahara Stadium → now MCA Stadium Pune
    'Subrata Roy Sahara Stadium': 'MCA Stadium Pune',

    # Shaheed Veer Narayan Singh Stadium, Raipur
    'Shaheed Veer Narayan Singh International Stadium, Raipur': 'SVNS Stadium Raipur',
    'Shaheed Veer Narayan Singh International Stadium': 'SVNS Stadium Raipur',

    # Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': 'Ekana Stadium Lucknow',

    # Saurashtra Cricket Association Stadium
    'Saurashtra Cricket Association Stadium': 'SCA Stadium Rajkot',

    # Zayed Cricket Stadium, Abu Dhabi
    'Zayed Cricket Stadium, Abu Dhabi': 'Sheikh Zayed Stadium',

    # Barsapara Cricket Stadium, Guwahati
    'Barsapara Cricket Stadium, Guwahati': 'Barsapara Stadium Guwahati',

    # Vidarbha Cricket Association Stadium
    'Vidarbha Cricket Association Stadium, Jamtha': 'VCA Stadium Nagpur',

    # JSCA International Stadium Complex
    'JSCA International Stadium Complex': 'JSCA Stadium Ranchi',
}

# Apply mapping
df['venue'] = df['venue'].replace(venue_mapping)

# Verify — check unique venues after standardization
venues_after = df['venue'].value_counts().reset_index()
venues_after.columns = ['venue', 'deliveries']
print(f"Venues before: 60")
print(f"Venues after standardization: {len(venues_after)}")
print(f"\nAll standardized venues:")
print(venues_after.to_string())

In [ ]:
# Save updated cleaned data with standardized venues
df.to_csv(r"D:\COEP\Cricket_Project\data\processed\ipl_cleaned.csv", index=False)
print("✓ Updated data saved with standardized venues")

In [ ]:
# === INSIGHT 5: Venue Analysis — Batting vs Bowling Grounds ===

venue_stats = df.groupby('venue').agg(
    total_runs    = ('total_runs', 'sum'),
    total_balls   = ('is_legal', 'sum'),
    total_wickets = ('is_wicket', 'sum'),
    total_matches = ('match_id', 'nunique')
).reset_index()

# Minimum 20 matches
venue_stats = venue_stats[venue_stats['total_matches'] >= 20].copy()

venue_stats['avg_runs_per_match'] = (venue_stats['total_runs'] / venue_stats['total_matches']).round(1)
venue_stats['economy']            = (venue_stats['total_runs'] / venue_stats['total_balls'] * 6).round(2)
venue_stats['wickets_per_match']  = (venue_stats['total_wickets'] / venue_stats['total_matches']).round(2)

# Batting Bias Score
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
venue_stats['run_score']    = scaler.fit_transform(venue_stats[['avg_runs_per_match']])
venue_stats['wicket_score'] = scaler.fit_transform(venue_stats[['wickets_per_match']])
venue_stats['batting_bias'] = (
    0.6 * venue_stats['run_score'] +
    0.4 * (1 - venue_stats['wicket_score'])
).round(3)

venue_sorted = venue_stats.sort_values('batting_bias', ascending=True)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('IPL Venue Intelligence — Batting vs Bowling Grounds',
             fontsize=14, fontweight='bold')

# Left — Batting Bias Bar Chart
bar_colors = ['#E8612C' if b >= 0.6 else '#3498db' if b <= 0.4 else '#95a5a6'
              for b in venue_sorted['batting_bias']]
ax1.barh(venue_sorted['venue'], venue_sorted['batting_bias'],
         color=bar_colors, edgecolor='white', height=0.6)
ax1.axvline(x=0.5, color='black', linestyle='--',
            linewidth=1.5, label='Neutral (0.5)')
ax1.set_xlabel('Batting Bias Score (Higher = More batting friendly)', fontsize=11)
ax1.set_title('Venue Bias Score\nRed = Batting Paradise | Grey = Neutral | Blue = Bowling Ground', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, linestyle='--', axis='x')

# Right — Economy vs Avg Score Scatter
scatter = ax2.scatter(
    venue_stats['economy'],
    venue_stats['avg_runs_per_match'],
    s = venue_stats['total_matches'] * 4,
    c = venue_stats['batting_bias'].values,
    cmap = 'RdYlGn',
    alpha = 0.8,
    edgecolors = 'white',
    linewidth = 0.5
)

for _, row in venue_stats.iterrows():
    ax2.annotate(row['venue'].split()[0],
                xy=(row['economy'], row['avg_runs_per_match']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=7, alpha=0.8)

plt.colorbar(scatter, ax=ax2, label='Batting Bias Score')
ax2.set_xlabel('Overall Economy (runs per over)', fontsize=11)
ax2.set_ylabel('Average Runs Per Match', fontsize=11)
ax2.set_title('Economy vs Avg Score per Venue\n(bubble size = matches played)', fontsize=11)
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(r'D:\COEP\Cricket_Project\outputs\05_venue_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("Top 5 Batting Paradises:")
print(venue_stats.nlargest(5, 'batting_bias')[
    ['venue','avg_runs_per_match','economy','batting_bias',
     'total_matches']].to_string(index=False))

print("\nTop 5 Bowling Grounds:")
print(venue_stats.nsmallest(5, 'batting_bias')[
    ['venue','avg_runs_per_match','economy','batting_bias',
     'total_matches']].to_string(index=False))

In [ ]:
# === INSIGHT 6: Team Performance Trends ===

# Step 1 — Match level results from info files
import os
import glob

info_files = glob.glob(r"D:\COEP\Cricket_Project\data\raw\*_info.csv")
print(f"Info files found: {len(info_files)}")

# Load all info files
info_dfs = []
for f in info_files:
    try:
        temp = pd.read_csv(f, header=None, names=['type','value1','value2','value3'])
        match_id = os.path.basename(f).replace("_info.csv","")
        temp['match_id'] = match_id
        info_dfs.append(temp)
    except:
        pass

info_df = pd.concat(info_dfs, ignore_index=True)
print(f"Info records loaded: {len(info_df):,}")
print(f"\nInfo types available:")
print(info_df['type'].value_counts().head(15))

In [ ]:
# === Load info files correctly ===

info_dfs = []
for f in info_files:
    try:
        temp = pd.read_csv(f, header=None)
        # Keep only rows where first column is 'info'
        temp = temp[temp[0] == 'info'].copy()
        temp.columns = ['type', 'key', 'value'] + list(range(temp.shape[1]-3))
        temp = temp[['key','value']].copy()
        match_id = os.path.basename(f).replace("_info.csv","")
        temp['match_id'] = match_id
        info_dfs.append(temp)
    except:
        pass

info_df = pd.concat(info_dfs, ignore_index=True)
print(f"Info records loaded: {len(info_df):,}")
print(f"\nUnique keys:")
print(info_df['key'].value_counts())

In [ ]:
# Parse info files using raw text — handles variable columns
info_dfs = []

for f in info_files:
    try:
        match_id = os.path.basename(f).replace("_info.csv","")
        rows = []
        with open(f, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if line.startswith('info,'):
                    parts = line.split(',', 2)  # split max 2 times
                    if len(parts) >= 3:
                        rows.append({
                            'key': parts[1].strip(),
                            'value': parts[2].strip(),
                            'match_id': match_id
                        })
        if rows:
            info_dfs.append(pd.DataFrame(rows))
    except Exception as e:
        print(f"Error in {f}: {e}")

info_df = pd.concat(info_dfs, ignore_index=True)
print(f"✓ Info records loaded: {len(info_df):,}")
print(f"\nUnique keys:")
print(info_df['key'].value_counts())

In [ ]:
# === Extract match results ===

# Pivot info_df to get one row per match
winner_df = info_df[info_df['key'] == 'winner'][['match_id','value']].copy()
winner_df.columns = ['match_id', 'winner']

season_df = info_df[info_df['key'] == 'season'][['match_id','value']].copy()
season_df.columns = ['match_id', 'season']
season_df['season'] = season_df['season'].astype(str).str[:4].astype(int)

team_df = info_df[info_df['key'] == 'team'].groupby('match_id')['value'].apply(list).reset_index()
team_df.columns = ['match_id', 'teams']

# Merge all
match_df = winner_df.merge(season_df, on='match_id').merge(team_df, on='match_id')
print(f"Matches with results: {len(match_df)}")
print(match_df.head(3))

In [ ]:
# === Team Performance Analysis ===

# Expand teams — each match has 2 teams
# Create one row per team per match
team_records = []

for _, row in match_df.iterrows():
    for team in row['teams']:
        team_records.append({
            'match_id': row['match_id'],
            'season':   row['season'],
            'team':     team,
            'won':      1 if team == row['winner'] else 0
        })

team_match_df = pd.DataFrame(team_records)
print(f"Team-match records: {len(team_match_df)}")

# Overall team stats
team_stats = team_match_df.groupby('team').agg(
    matches = ('match_id', 'count'),
    wins    = ('won', 'sum')
).reset_index()

team_stats['win_rate'] = (team_stats['wins'] / team_stats['matches'] * 100).round(1)
team_stats = team_stats[team_stats['matches'] >= 30].sort_values('win_rate', ascending=False)

print(f"\nTeam Win Rates (min 30 matches):")
print(team_stats.to_string(index=False))

In [ ]:
# === Team Name Standardization ===

team_mapping = {
    # RCB renamed
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore',
    
    # Delhi renamed
    'Delhi Daredevils': 'Delhi Capitals',
    
    # Punjab renamed
    'Kings XI Punjab': 'Punjab Kings',
    
    # Pune teams (defunct — keep separate, they were different franchises)
    # Rising Pune Supergiant — keep as is
    # Pune Warriors — keep as is

    # Deccan Chargers → became Sunrisers Hyderabad (different owner, keep separate)
}

# Apply to both dataframes
team_match_df['team']        = team_match_df['team'].replace(team_mapping)
team_match_df['team']        = team_match_df['team'].replace(team_mapping)
match_df['winner']           = match_df['winner'].replace(team_mapping)
info_df['value']             = info_df['value'].replace(team_mapping)

# Also apply to main df
df['batting_team']           = df['batting_team'].replace(team_mapping)
df['bowling_team']           = df['bowling_team'].replace(team_mapping)

# Recalculate team stats
team_stats = team_match_df.groupby('team').agg(
    matches = ('match_id', 'count'),
    wins    = ('won', 'sum')
).reset_index()

team_stats['win_rate'] = (team_stats['wins'] / team_stats['matches'] * 100).round(1)
team_stats = team_stats[team_stats['matches'] >= 30].sort_values('win_rate', ascending=False)

print("Team Win Rates after standardization:")
print(team_stats.to_string(index=False))

In [ ]:
# Recalculate from scratch after standardization
team_records = []

for _, row in match_df.iterrows():
    for team in row['teams']:
        # Apply standardization here directly
        team_name = team_mapping.get(team, team)
        winner_name = team_mapping.get(row['winner'], row['winner'])
        team_records.append({
            'match_id': row['match_id'],
            'season':   row['season'],
            'team':     team_name,
            'won':      1 if team_name == winner_name else 0
        })

team_match_df = pd.DataFrame(team_records)

# Recalculate
team_stats = team_match_df.groupby('team').agg(
    matches = ('match_id', 'count'),
    wins    = ('won', 'sum')
).reset_index()

team_stats['win_rate'] = (team_stats['wins'] / team_stats['matches'] * 100).round(1)
team_stats = team_stats[team_stats['matches'] >= 30].sort_values('win_rate', ascending=False)

print("Team Win Rates (corrected):")
print(team_stats.to_string(index=False))

In [ ]:
# === INSIGHT 6: Team Performance Visualization ===

# Season wise win rate per team
season_team = team_match_df.groupby(['season','team']).agg(
    matches = ('match_id','count'),
    wins    = ('won','sum')
).reset_index()
season_team['win_rate'] = (season_team['wins'] / season_team['matches'] * 100).round(1)

# Only current/major teams
major_teams = ['Chennai Super Kings','Mumbai Indians','Kolkata Knight Riders',
               'Royal Challengers Bangalore','Rajasthan Royals','Delhi Capitals',
               'Sunrisers Hyderabad','Punjab Kings']

season_team_filtered = season_team[season_team['team'].isin(major_teams)]

# Team colors
team_colors = {
    'Chennai Super Kings':       '#F9CD05',
    'Mumbai Indians':            '#004BA0',
    'Kolkata Knight Riders':     '#3A225D',
    'Royal Challengers Bangalore':'#EC1C24',
    'Rajasthan Royals':          '#EA1A85',
    'Delhi Capitals':            '#0078BC',
    'Sunrisers Hyderabad':       '#FF822A',
    'Punjab Kings':              '#808080',
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))
fig.suptitle('IPL Team Performance Analysis', fontsize=16, fontweight='bold')

# Top chart — Overall win rate bar chart
top_teams = team_stats[team_stats['team'].isin(major_teams)].sort_values('win_rate', ascending=True)
colors_bar = [team_colors.get(t, '#95a5a6') for t in top_teams['team']]

bars = ax1.barh(top_teams['team'], top_teams['win_rate'],
                color=colors_bar, edgecolor='white', height=0.6)

# Add win/loss labels
for bar, (_, row) in zip(bars, top_teams.iterrows()):
    ax1.text(bar.get_width() + 0.5,
             bar.get_y() + bar.get_height()/2,
             f"{row['win_rate']}%  ({row['wins']}W / {row['matches']}M)",
             va='center', fontsize=9)

ax1.axvline(x=50, color='black', linestyle='--', linewidth=1.5, label='50% win rate')
ax1.set_xlabel('Win Rate %', fontsize=11)
ax1.set_title('Overall IPL Win Rate — All Seasons Combined', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 75)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, linestyle='--', axis='x')

# Bottom chart — Season wise win rate trends
for team in major_teams:
    team_data = season_team_filtered[
        (season_team_filtered['team'] == team) &
        (season_team_filtered['matches'] >= 5)
    ].sort_values('season')

    if len(team_data) > 2:
        ax2.plot(team_data['season'], team_data['win_rate'],
                marker='o', linewidth=2, markersize=5,
                label=team.replace('Royal Challengers Bangalore','RCB')
                         .replace('Chennai Super Kings','CSK')
                         .replace('Mumbai Indians','MI')
                         .replace('Kolkata Knight Riders','KKR')
                         .replace('Rajasthan Royals','RR')
                         .replace('Delhi Capitals','DC')
                         .replace('Sunrisers Hyderabad','SRH')
                         .replace('Punjab Kings','PBKS'),
                color=team_colors.get(team, '#95a5a6'),
                alpha=0.85)

ax2.axhline(y=50, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Season', fontsize=11)
ax2.set_ylabel('Win Rate %', fontsize=11)
ax2.set_title('Season-wise Win Rate Trends (2008-2026)', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9, ncol=2)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.set_xticks(season_team_filtered['season'].unique())
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(r'D:\COEP\Cricket_Project\outputs\06_team_performance.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Finding:")
print(f"Most consistent team: {team_stats[team_stats['team'].isin(major_teams)].nlargest(1,'win_rate')['team'].values[0]}")
print(f"Most matches played: {team_stats.nlargest(1,'matches')['team'].values[0]}")

In [ ]:
# === INSIGHT 7: Pressure Index ===

# Step 1 — Build cumulative match state per delivery
# Sort data properly first
df = df.sort_values(['match_id','innings','over_number','ball_number']).reset_index(drop=True)

# Cumulative runs and wickets within each innings
df['cum_runs'] = df.groupby(['match_id','innings'])['total_runs'].cumsum()
df['cum_wickets'] = df.groupby(['match_id','innings'])['is_wicket'].cumsum()

# Shift by 1 — we want state BEFORE this delivery, not after
df['runs_so_far'] = df.groupby(['match_id','innings'])['cum_runs'].shift(1).fillna(0)
df['wickets_so_far'] = df.groupby(['match_id','innings'])['cum_wickets'].shift(1).fillna(0)

print("✓ Cumulative state built")
print(df[['match_id','innings','over_number','ball_number',
          'runs_so_far','wickets_so_far','total_runs']].head(15))

In [ ]:
# Step 2 — Get first innings total (this becomes target for 2nd innings)
first_innings_total = df[df['innings'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
first_innings_total.columns = ['match_id', 'first_innings_score']

# Merge target into main df
df = df.merge(first_innings_total, on='match_id', how='left')

# Step 3 — Pressure Index formula
def compute_pressure(row):
    over = row['over_number']
    wickets = row['wickets_so_far']
    
    # Wicket pressure — normalized 0 to 1
    wicket_pressure = wickets / 10.0
    
    # Over pressure — later overs = more pressure
    over_pressure = over / 19.0
    
    if row['innings'] == 2:
        # Chase pressure — based on required run rate
        runs_scored   = row['runs_so_far']
        target        = row['first_innings_score'] + 1
        balls_bowled  = (over * 6) + row['ball_number']
        balls_remaining = max((20 * 6) - balls_bowled, 1)
        runs_needed   = max(target - runs_scored, 0)
        
        # Required run rate per ball
        rrr_per_ball  = runs_needed / balls_remaining
        # Normalize — 3 runs/ball (18 rpo) = maximum pressure
        rrr_pressure  = min(rrr_per_ball / 3.0, 1.0)
        
        pressure = (
            0.50 * rrr_pressure    +   # chase situation dominates
            0.30 * wicket_pressure +   # wickets lost
            0.20 * over_pressure       # how late in innings
        )
    else:
        # First innings — no explicit target pressure
        pressure = (
            0.50 * wicket_pressure +   # wickets matter more
            0.50 * over_pressure       # and over number
        )
    
    return round(pressure * 100, 2)

# Apply — this will take 1-2 minutes on 294K rows
from tqdm.notebook import tqdm
tqdm.pandas(desc="Computing Pressure Index")
df['pressure_index'] = df.progress_apply(compute_pressure, axis=1)

print("✓ Pressure Index computed")
print(f"\nPressure Index stats:")
print(df['pressure_index'].describe().round(2))

In [ ]:
# === Pressure Index Visualization ===

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Pressure Index Analysis — IPL Ball by Ball',
             fontsize=16, fontweight='bold')

# Chart 1 — Pressure distribution
ax1 = axes[0, 0]
ax1.hist(df['pressure_index'], bins=50, color='#E8612C',
         edgecolor='white', alpha=0.85)
ax1.axvline(df['pressure_index'].mean(), color='black',
            linestyle='--', linewidth=2,
            label=f"Mean: {df['pressure_index'].mean():.1f}")
ax1.set_xlabel('Pressure Index (0-100)', fontsize=11)
ax1.set_ylabel('Number of Deliveries', fontsize=11)
ax1.set_title('Distribution of Pressure Across All Deliveries', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, linestyle='--')

# Chart 2 — Average pressure by over (both innings)
ax2 = axes[0, 1]
pressure_by_over = df.groupby(['innings', 'over_number'])['pressure_index'].mean().reset_index()

for innings_num, color, label in [(1, '#3498db', '1st Innings'),
                                   (2, '#E8612C', '2nd Innings (Chase)')]:
    data = pressure_by_over[pressure_by_over['innings'] == innings_num]
    ax2.plot(data['over_number'], data['pressure_index'],
             color=color, linewidth=2.5, marker='o',
             markersize=4, label=label)

ax2.set_xlabel('Over Number', fontsize=11)
ax2.set_ylabel('Average Pressure Index', fontsize=11)
ax2.set_title('How Pressure Builds Across Overs', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.set_xticks(range(0, 20))

# Chart 3 — Batters who face highest average pressure
ax3 = axes[1, 0]
batter_pressure = df.groupby('striker').agg(
    avg_pressure = ('pressure_index', 'mean'),
    total_balls  = ('is_legal', 'sum'),
    total_runs   = ('runs_off_bat', 'sum')
).reset_index()

# Min 200 balls
batter_pressure = batter_pressure[(batter_pressure['total_balls'] >= 200) & (batter_pressure['total_runs'] >= 1500)]
batter_pressure['sr'] = (batter_pressure['total_runs'] /
                          batter_pressure['total_balls'] * 100).round(1)

# Top 15 by avg pressure faced

top_pressure = batter_pressure.nlargest(15, 'avg_pressure')

ax3.barh(top_pressure['striker'], top_pressure['avg_pressure'],
         color='#9b59b6', edgecolor='white', height=0.6)
ax3.set_xlabel('Average Pressure Index Faced', fontsize=11)
ax3.set_title('Batters Who Face Most Pressure\n(Min 200 balls and Min 1500 runs)', fontsize=12)
ax3.invert_yaxis()
ax3.grid(True, alpha=0.3, linestyle='--', axis='x')

# Chart 4 — Strike rate under high vs low pressure
ax4 = axes[1, 1]

# High pressure = index > 60, Low pressure = index < 30
high_p = df[df['pressure_index'] > 60].groupby('striker').agg(
    runs  = ('runs_off_bat', 'sum'),
    balls = ('is_legal', 'sum')
).reset_index()
high_p = high_p[high_p['balls'] >= 50]
high_p['high_pressure_sr'] = (high_p['runs'] / high_p['balls'] * 100).round(1)

low_p = df[df['pressure_index'] < 30].groupby('striker').agg(
    runs  = ('runs_off_bat', 'sum'),
    balls = ('is_legal', 'sum')
).reset_index()
low_p = low_p[low_p['balls'] >= 50]
low_p['low_pressure_sr'] = (low_p['runs'] / low_p['balls'] * 100).round(1)

# Merge
pressure_compare = high_p[['striker','high_pressure_sr']].merge(
    low_p[['striker','low_pressure_sr']], on='striker')
pressure_compare['pressure_drop'] = (
    pressure_compare['low_pressure_sr'] -
    pressure_compare['high_pressure_sr']).round(1)

# Players whose SR drops LEAST under pressure = most clutch
clutch_players = pressure_compare.nsmallest(12, 'pressure_drop')

x = range(len(clutch_players))
width = 0.35
ax4.bar([i - width/2 for i in x], clutch_players['low_pressure_sr'],
        width, label='Low Pressure SR', color='#2ecc71', edgecolor='white')
ax4.bar([i + width/2 for i in x], clutch_players['high_pressure_sr'],
        width, label='High Pressure SR', color='#E8612C', edgecolor='white')

ax4.set_xticks(list(x))
ax4.set_xticklabels([name.split()[-1] for name in clutch_players['striker']],
                     rotation=45, ha='right', fontsize=9)
ax4.set_ylabel('Strike Rate', fontsize=11)
ax4.set_title('Most Clutch Batters\n(Smallest SR drop under pressure)', fontsize=12)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3, linestyle='--', axis='y')

plt.tight_layout()
plt.savefig(r'D:\COEP\Cricket_Project\outputs\07_pressure_index.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 Most Clutch Batters (least SR drop under pressure):")
print(pressure_compare.nsmallest(10,'pressure_drop')[
    ['striker','low_pressure_sr','high_pressure_sr','pressure_drop']
].to_string(index=False))

In [ ]:
# Save with pressure index
df.to_csv(r"D:\COEP\Cricket_Project\data\processed\ipl_cleaned.csv", index=False)
print(f"✓ Saved — columns now: {df.columns.tolist()}")

In [ ]:
# === INSIGHT 8: Bowler Performance Under Pressure ===

# Stricter filters — min 200 high pressure balls, min 300 low pressure balls
high_p_bowl = df[df['pressure_index'] > 60].groupby('bowler').agg(
    runs_conceded  = ('total_runs', 'sum'),
    balls          = ('is_legal', 'sum'),
    wickets        = ('is_wicket', 'sum'),
    dot_balls      = ('total_runs', lambda x: (x == 0).sum())
).reset_index()

# Increased minimum balls
high_p_bowl = high_p_bowl[high_p_bowl['balls'] >= 200].copy()
high_p_bowl['high_economy']  = (high_p_bowl['runs_conceded'] / high_p_bowl['balls'] * 6).round(2)
high_p_bowl['high_dot_pct']  = (high_p_bowl['dot_balls'] / high_p_bowl['balls'] * 100).round(1)
high_p_bowl['high_wkt_rate'] = (high_p_bowl['wickets'] / high_p_bowl['balls'] * 6).round(3)

low_p_bowl = df[df['pressure_index'] < 30].groupby('bowler').agg(
    runs_conceded  = ('total_runs', 'sum'),
    balls          = ('is_legal', 'sum'),
    wickets        = ('is_wicket', 'sum')
).reset_index()
low_p_bowl = low_p_bowl[low_p_bowl['balls'] >= 300].copy()
low_p_bowl['low_economy'] = (low_p_bowl['runs_conceded'] / low_p_bowl['balls'] * 6).round(2)

# Merge — inner join ensures bowler has enough balls in BOTH conditions
bowl_compare = high_p_bowl.merge(low_p_bowl[['bowler','low_economy']], on='bowler')

# Sanity check — economy should be realistic (between 5 and 13)
bowl_compare = bowl_compare[
    (bowl_compare['low_economy'].between(5, 13)) &
    (bowl_compare['high_economy'].between(5, 13))
].copy()

bowl_compare['economy_change'] = (bowl_compare['high_economy'] -
                                   bowl_compare['low_economy']).round(2)

# Pressure Bowling Score
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

bowl_compare['econ_score'] = 1 - scaler.fit_transform(bowl_compare[['economy_change']])
bowl_compare['dot_score']  = scaler.fit_transform(bowl_compare[['high_dot_pct']])
bowl_compare['wkt_score']  = scaler.fit_transform(bowl_compare[['high_wkt_rate']])

bowl_compare['pressure_bowling_score'] = (
    0.40 * bowl_compare['econ_score'] +
    0.35 * bowl_compare['dot_score']  +
    0.25 * bowl_compare['wkt_score']
).round(3)

top30_bowl = bowl_compare.nlargest(30, 'pressure_bowling_score')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 14))  # taller figure for 30 names
fig.suptitle('Bowler Performance Under Pressure (Pressure Index > 60)',
             fontsize=14, fontweight='bold')

colors = ['#2ecc71' if i < 5 else '#3498db' if i < 15 else '#95a5a6'
          for i in range(len(top30_bowl))]

ax1.barh(top30_bowl['bowler'], top30_bowl['pressure_bowling_score'],
         color=colors, edgecolor='white', height=0.6)
ax1.set_xlabel('Pressure Bowling Score (0-1)', fontsize=11)
ax1.set_title('Top 30 Pressure Bowlers\n(Min 200 high-pressure balls)', fontsize=11)
ax1.invert_yaxis()
ax1.grid(True, alpha=0.3, linestyle='--', axis='x')

for i, (_, row) in enumerate(top30_bowl.iterrows()):
    ax1.text(row['pressure_bowling_score'] + 0.005,
             i, f"{row['pressure_bowling_score']:.3f}",
             va='center', fontsize=8)

# Right chart — top 10 economy comparison (keep readable)
top10 = bowl_compare.nlargest(10, 'pressure_bowling_score')
x     = range(len(top10))
width = 0.35

ax2.bar([i - width/2 for i in x], top10['low_economy'],
        width, label='Low Pressure Economy',
        color='#2ecc71', edgecolor='white')
ax2.bar([i + width/2 for i in x], top10['high_economy'],
        width, label='High Pressure Economy',
        color='#E8612C', edgecolor='white')

ax2.set_xticks(list(x))
ax2.set_xticklabels([name.split()[-1] for name in top10['bowler']],
                     rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('Economy Rate', fontsize=11)
ax2.set_title('Economy: Low vs High Pressure\n(Top 10 Pressure Bowlers)', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')

plt.tight_layout()
plt.savefig(r'D:\COEP\Cricket_Project\outputs\08_bowler_pressure.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"Bowlers qualifying: {len(bowl_compare)}")
print("\nTop 30 Pressure Bowlers:")
print(top30_bowl[['bowler','low_economy','high_economy',
                   'economy_change','high_dot_pct',
                   'pressure_bowling_score']].to_string(index=False))

In [ ]:
# === INSIGHT 9: Head to Head — Batter vs Bowler ===

# All batter-bowler matchups
h2h = df[df['is_legal'] == 1].groupby(['striker','bowler']).agg(
    balls        = ('is_legal', 'sum'),
    runs         = ('runs_off_bat', 'sum'),
    dismissals   = ('is_wicket', 'sum'),
    dot_balls    = ('runs_off_bat', lambda x: (x == 0).sum()),
    fours        = ('runs_off_bat', lambda x: (x == 4).sum()),
    sixes        = ('runs_off_bat', lambda x: (x == 6).sum()),
).reset_index()

# Minimum 10 balls — meaningful matchup
h2h = h2h[h2h['balls'] >= 10].copy()

h2h['strike_rate']  = (h2h['runs'] / h2h['balls'] * 100).round(1)
h2h['dot_pct']      = (h2h['dot_balls'] / h2h['balls'] * 100).round(1)
h2h['boundary_pct'] = ((h2h['fours'] + h2h['sixes']) / h2h['balls'] * 100).round(1)

# Dominance score — from batter's perspective
# High SR + low dismissals = batter dominates
# Low SR + high dismissals = bowler dominates
h2h['batter_dominance'] = (
    (h2h['strike_rate'] / 100) -
    (h2h['dismissals'] / h2h['balls'] * 6)
).round(3)

print(f"Total meaningful matchups (min 10 balls): {len(h2h):,}")
print(f"\nSample — Rohit Sharma vs various bowlers:")

rohit = h2h[h2h['striker'] == 'RG Sharma'].sort_values('balls', ascending=False)
print(rohit[['striker','bowler','balls','runs','strike_rate',
             'dismissals','dot_pct','batter_dominance']].head(10).to_string(index=False))

In [ ]:
# === H2H Visualization + Lookup Function ===

import matplotlib.patches as mpatches

def plot_h2h(batter_name, top_n=10):
    """
    Given a batter name — show their best and worst bowler matchups
    """
    batter_data = h2h[h2h['striker'] == batter_name].copy()
    
    if len(batter_data) == 0:
        print(f"No data found for {batter_name}")
        print("Available batters sample:", h2h['striker'].unique()[:20])
        return
    
    # Min 20 balls for visualization
    batter_data = batter_data[batter_data['balls'] >= 20].sort_values(
        'batter_dominance', ascending=False)
    
    if len(batter_data) == 0:
        print(f"Not enough data for {batter_name} (need min 20 balls per matchup)")
        return
    
    best_matchups  = batter_data.head(top_n)    # batter dominates
    worst_matchups = batter_data.tail(top_n)     # bowler dominates
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'Head to Head Analysis — {batter_name}',
                 fontsize=15, fontweight='bold')
    
    # Left — Best matchups (batter dominates)
    ax1 = axes[0]
    colors_best = ['#2ecc71'] * len(best_matchups)
    bars = ax1.barh(best_matchups['bowler'], best_matchups['strike_rate'],
                    color=colors_best, edgecolor='white', height=0.6)
    for bar, (_, row) in zip(bars, best_matchups.iterrows()):
        ax1.text(bar.get_width() + 1,
                 bar.get_y() + bar.get_height()/2,
                 f"SR:{row['strike_rate']} | {row['runs']}R {row['dismissals']}W ({row['balls']}b)",
                 va='center', fontsize=8)
    ax1.set_xlabel('Strike Rate', fontsize=11)
    ax1.set_title(f'Batters best matchups\n(Dominates these bowlers)', fontsize=11)
    ax1.invert_yaxis()
    ax1.set_xlim(0, best_matchups['strike_rate'].max() * 1.5)
    ax1.grid(True, alpha=0.3, linestyle='--', axis='x')
    
    # Middle — Worst matchups (bowler dominates)
    ax2 = axes[1]
    worst_sorted = worst_matchups.sort_values('batter_dominance', ascending=True)
    colors_worst = ['#E8612C'] * len(worst_sorted)
    bars2 = ax2.barh(worst_sorted['bowler'], worst_sorted['strike_rate'],
                     color=colors_worst, edgecolor='white', height=0.6)
    for bar, (_, row) in zip(bars2, worst_sorted.iterrows()):
        ax2.text(bar.get_width() + 1,
                 bar.get_y() + bar.get_height()/2,
                 f"SR:{row['strike_rate']} | {row['runs']}R {row['dismissals']}W ({row['balls']}b)",
                 va='center', fontsize=8)
    ax2.set_xlabel('Strike Rate', fontsize=11)
    ax2.set_title(f'Batters worst matchups\n(Struggles against these bowlers)', fontsize=11)
    ax2.invert_yaxis()
    ax2.set_xlim(0, worst_sorted['strike_rate'].max() * 1.5)
    ax2.grid(True, alpha=0.3, linestyle='--', axis='x')
    
    # Right — Dominance score overview (all matchups)
    ax3 = axes[2]
    plot_data = batter_data.sort_values('batter_dominance').tail(20)
    colors_dom = ['#2ecc71' if d > 0.8 else '#E8612C' if d < 0.5 else '#f39c12'
                  for d in plot_data['batter_dominance']]
    ax3.barh(plot_data['bowler'], plot_data['batter_dominance'],
             color=colors_dom, edgecolor='white', height=0.6)
    ax3.axvline(x=1.0, color='black', linestyle='--',
                linewidth=1.5, label='Balanced (1.0)', alpha=0.7)
    ax3.set_xlabel('Batter Dominance Score', fontsize=11)
    ax3.set_title('Dominance Score vs All Bowlers\n(>1.0 = batter wins)', fontsize=11)
    
    green_patch  = mpatches.Patch(color='#2ecc71', label='Batter dominates')
    orange_patch = mpatches.Patch(color='#f39c12', label='Balanced')
    red_patch    = mpatches.Patch(color='#E8612C', label='Bowler dominates')
    ax3.legend(handles=[green_patch, orange_patch, red_patch], fontsize=9)
    ax3.grid(True, alpha=0.3, linestyle='--', axis='x')
    
    plt.tight_layout()
    
    save_name = batter_name.replace(' ','_').replace('.','')
    plt.savefig(
        fr'D:\COEP\Cricket_Project\outputs\09_h2h_{save_name}.png',
        dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print summary
    print(f"\n{batter_name} — Head to Head Summary")
    print(f"Total meaningful matchups: {len(batter_data)}")
    print(f"\nTop 5 bowlers he DOMINATES:")
    print(batter_data.head(5)[['bowler','balls','runs',
                                'strike_rate','dismissals']].to_string(index=False))
    print(f"\nTop 5 bowlers who DOMINATE him:")
    print(batter_data.tail(5).sort_values('batter_dominance')[
        ['bowler','balls','runs','strike_rate','dismissals']].to_string(index=False))

# =====================
# Try different players
# =====================
plot_h2h('RG Sharma')
plot_h2h('V Kohli')
plot_h2h('MS Dhoni')
plot_h2h('AB de Villiers')


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from sklearn.preprocessing import MinMaxScaler

# ── Load all processed data ──────────────────────────────────
base = r"D:\COEP\Cricket_Project\data\processed"

df            = pd.read_csv(f"{base}\\ipl_cleaned.csv", parse_dates=['start_date'])
season_stats  = pd.read_csv(f"{base}\\season_stats.csv")
venue_stats   = pd.read_csv(f"{base}\\venue_stats.csv")
team_stats    = pd.read_csv(f"{base}\\team_stats.csv")
bowl_compare  = pd.read_csv(f"{base}\\bowling_stats.csv")
h2h           = pd.read_csv(f"{base}\\h2h_stats.csv")

print("✓ All data loaded")
print(f"  Main df     : {len(df):,} rows")
print(f"  Season stats: {len(season_stats)} seasons")
print(f"  Venue stats : {len(venue_stats)} venues")
print(f"  Team stats  : {len(team_stats)} teams")
print(f"  Bowl compare: {len(bowl_compare)} bowlers")
print(f"  H2H matchups: {len(h2h):,} matchups")
batting_stats = pd.read_csv(f"{base}\\batting_stats.csv")
print(f"  Batting stats: {len(batting_stats)} players")

✓ All data loaded
  Main df     : 294,757 rows
  Season stats: 18 seasons
  Venue stats : 16 venues
  Team stats  : 12 teams
  Bowl compare: 74 bowlers
  H2H matchups: 9,007 matchups
  Batting stats: 736 players


In [5]:
# Build batting stats
batting_stats = df[df['is_legal'] == 1].groupby('striker').agg(
    innings      = ('match_id', 'nunique'),
    total_runs   = ('runs_off_bat', 'sum'),
    balls_faced  = ('is_legal', 'sum'),
    fours        = ('runs_off_bat', lambda x: (x == 4).sum()),
    sixes        = ('runs_off_bat', lambda x: (x == 6).sum()),
    dismissals   = ('is_wicket', 'sum'),
    avg_pressure = ('pressure_index', 'mean')
).reset_index()

batting_stats['strike_rate']   = (batting_stats['total_runs'] / batting_stats['balls_faced'] * 100).round(2)
batting_stats['average']       = (batting_stats['total_runs'] / batting_stats['dismissals'].replace(0,1)).round(2)
batting_stats['boundary_rate'] = ((batting_stats['fours'] + batting_stats['sixes']) / batting_stats['balls_faced'] * 100).round(2)

# Save
batting_stats.to_csv(r"D:\COEP\Cricket_Project\data\processed\batting_stats.csv", index=False)
print(f"Batting stats saved — {len(batting_stats)} players")
print(batting_stats[batting_stats['innings'] >= 20].nlargest(5,'strike_rate')[
    ['striker','innings','total_runs','strike_rate','average']].to_string(index=False))

Batting stats saved — 736 players
      striker  innings  total_runs  strike_rate  average
V Suryavanshi       21         835       224.46    39.76
Priyansh Arya       31         909       193.82    29.32
     TH David       57        1105       178.23    33.48
      PD Salt       40        1253       175.24    32.97
   AD Russell      114        2632       174.54    28.61


In [6]:
# Final save — saari insights ke baad
df.to_csv(r"D:\COEP\Cricket_Project\data\processed\ipl_cleaned.csv", index=False)

# Also save key derived tables — dashboard mein direct use honge
batting_stats.to_csv(r"D:\COEP\Cricket_Project\data\processed\batting_stats.csv", index=False)
bowl_compare.to_csv(r"D:\COEP\Cricket_Project\data\processed\bowling_stats.csv", index=False)
venue_stats.to_csv(r"D:\COEP\Cricket_Project\data\processed\venue_stats.csv", index=False)
season_stats.to_csv(r"D:\COEP\Cricket_Project\data\processed\season_stats.csv", index=False)
team_stats.to_csv(r"D:\COEP\Cricket_Project\data\processed\team_stats.csv", index=False)
h2h.to_csv(r"D:\COEP\Cricket_Project\data\processed\h2h_stats.csv", index=False)

print("All data saved successfully")
print("\nFiles in processed folder:")
for f in os.listdir(r"D:\COEP\Cricket_Project\data\processed"):
    size = os.path.getsize(rf"D:\COEP\Cricket_Project\data\processed\{f}") / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")

All data saved successfully

Files in processed folder:
  batting_stats.csv: 0.0 MB
  bowling_stats.csv: 0.0 MB
  h2h_stats.csv: 0.5 MB
  ipl_cleaned.csv: 54.6 MB
  season_stats.csv: 0.0 MB
  team_stats.csv: 0.0 MB
  venue_stats.csv: 0.0 MB


In [ ]:
batting_stats.head()

In [ ]:
dir()